# Langchain Agents
# install all packages

In [ ]:
# !pip install langchain-experimental openai watermark tabulate pandas
# !pip install --upgrade langchain
# !pip install langchain openai


# import all the required libraries

In [6]:
import os
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI



# setup the environment

In [37]:



# Create client
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("GPT_API_KEY")

# LangSmith tracing settings
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")


In [38]:
dataset_file_name ="/Users/chandra/Desktop/FSDS_GenAI_Training/FSDS_Classes/Agentic_AI/titanic.csv"

df = pd.read_csv(dataset_file_name)
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [39]:
llm = OpenAI(api_key= os.getenv("GPT_API_KEY"))
from langchain_openai import ChatOpenAI

In [40]:
llm = ChatOpenAI(temperature=1,
                 model=os.getenv("GPT_MODEL"),
                 model_kwargs={"reasoning_effort": "none"})

In [41]:
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    agent_type="openai-tools",# or "Zero shot react description"
    allow_dangerous_code=True # this is the critical fix
)

In [42]:
agent.run("How many row are there?")



> Entering new AgentExecutor chain...
There are **891 rows** in `df`.

> Finished chain.


'There are **891 rows** in `df`.'

In [43]:
agent.run("How many male and female passengers survived")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Survived'] == 1].groupby('Sex').size()"}`


Sex
female    233
male      109
dtype: int64Among the survivors:

- **Female:** 233
- **Male:** 109

> Finished chain.


'Among the survivors:\n\n- **Female:** 233\n- **Male:** 109'

In [44]:
agent.run("How many people have more than 3 siblings?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.loc[df['SibSp'] > 3].shape[0]"}`


3030 people have more than 3 siblings.

> Finished chain.


'30 people have more than 3 siblings.'

In [45]:
agent.run("How many missing variables?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print(df.isna().sum())\nprint('\\nTotal missing values:', int(df.isna().sum().sum()))\nprint('Columns with missing values:', int(df.isna().any().sum()))"}`


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Total missing values: 866
Columns with missing values: 3
There are **3 variables (columns) with missing values**: `Age`, `Cabin`, and `Embarked`.

> Finished chain.


'There are **3 variables (columns) with missing values**: `Age`, `Cabin`, and `Embarked`.'

# Single Agent connecting to multiple data frames


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [49]:
df1 = df.copy()
df1["Age"] = df1["Age"].fillna(df1["Age"].mean())

In [ ]:
df1.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [54]:
# agent = create_pandas_dataframe_agent(llm, [df, df1], verbose=True)

agent = create_pandas_dataframe_agent(
    llm=llm,
    df=[df,df1],
    verbose=True,
    agent_type="openai-tools",# or "Zero shot react description"
    allow_dangerous_code=True # this is the critical fix
)

In [55]:
agent.run("How many rows in the age column are different")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print((df1['Age'].fillna('__NA__') != df2['Age'].fillna('__NA__')).sum())"}`


177
177 rows in the `Age` column are different between the two dataframes.

> Finished chain.


'177 rows in the `Age` column are different between the two dataframes.'

In [56]:
df2 = df1.copy()

In [57]:
df2.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [58]:
df2["Age_Multiplied"]=df1["Age"] * 2

In [59]:
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=[df,df1,df2],
    verbose=True,
    agent_type="openai-tools",# or "Zero shot react description"
    allow_dangerous_code=True # this is the critical fix
)

In [60]:
agent.run("Are the number of columns same in all the dataframe")



> Entering new AgentExecutor chain...
No. `df1` and `df2` have **13 columns**, while `df3` has **14 columns** because it includes the additional `Age_Multiplied` column.

> Finished chain.


'No. `df1` and `df2` have **13 columns**, while `df3` has **14 columns** because it includes the additional `Age_Multiplied` column.'